# Deep Learning - Homework 2

## Exploring Activation Functions on the Breast Cancer Wisconsin Dataset

### Marianna Kanellaki
### S-001081

In [1]:
#1-Batch & feature shapes, dtypes, and scaling

#2-Effect of scaling on the first projection (what many folks call the “input layer”)

#3-Categorical inputs: one‑hot vs embeddings (as an input layer)

#4-Images at input: HWC→CHW, normalization, flattening vs “ready for conv”

#5-Variable‑length sequences: padding + masks (input prep)

#6-Multi‑input merge: stitching different input types into one feature vector


### Data loading & Inspection

In [1]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['tumor type'] = data.target
df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,tumor type
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         569 non-null

In [4]:
print("\n📊 Descriptive Statistics:")
print(df.describe())

print("\n🔍 Missing Values:")
print(df.isnull().sum())


📊 Descriptive Statistics:
       mean radius  mean texture  mean perimeter    mean area  \
count   569.000000    569.000000      569.000000   569.000000   
mean     14.127292     19.289649       91.969033   654.889104   
std       3.524049      4.301036       24.298981   351.914129   
min       6.981000      9.710000       43.790000   143.500000   
25%      11.700000     16.170000       75.170000   420.300000   
50%      13.370000     18.840000       86.240000   551.100000   
75%      15.780000     21.800000      104.100000   782.700000   
max      28.110000     39.280000      188.500000  2501.000000   

       mean smoothness  mean compactness  mean concavity  mean concave points  \
count       569.000000        569.000000      569.000000           569.000000   
mean          0.096360          0.104341        0.088799             0.048919   
std           0.014064          0.052813        0.079720             0.038803   
min           0.052630          0.019380        0.000000       

The data do not contain any null values. All columns except the target are numerical with very different numbers. So we need to scale all values in the same range and encode the target that is categorical.

In [ ]:
df_original = df.copy()

#### Encoding categorical target

In [5]:
# Apply one-hot encoding to target
df_encoded = pd.get_dummies(df, columns=["tumor type"], drop_first=True)
df_encoded

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,tumor type_1
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,False
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,False
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,False
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,False
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,False
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,False
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,False
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,False


### Dataset Split

In [7]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(["tumor type_1"], axis=1)
y = df_encoded["tumor type_1"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Scaling

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Model Creation

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from keras.models import Sequential, Model
from keras.layers import Dense, Input
from keras.layers import LeakyReLU, ELU, GELU
from keras.optimizers import Adam
import tensorflow as tf

def build_model(activation_name, hidden_dim, num_hidden_layers=3):
    """ Creates a model with the specified activation function, initial hidden dimension, and number of hidden layers """
    
    inp = Input(shape=(2,), name="input_xy")
    x = inp

    for i in range(num_hidden_layers):
        if activation_name == "linear":
            x = Dense(hidden_dim, activation=None, name="hidden")(x)
        elif activation_name == "LeakyReLU(0.1)":
            z = Dense(hidden_dim, name="hidden_pre")(x)
            x = LeakyReLU(alpha=0.1, name="hidden")(z)
        elif activation_name == "ELU":
            z = Dense(hidden_dim, name="hidden_pre")(x)
            x = ELU(name="hidden")(z)
        else:
            x = Dense(hidden_dim, activation=f"{activation_name.lower()}", name="hidden")(x)

        hidden_dim //= 2

    out = Dense(1, activation="sigmoid", name="out")(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(1e-2), loss="binary_crossentropy", metrics=["accuracy"])

    return model

ModuleNotFoundError: No module named 'tensorflow.python'

### Activation Functions Experiment

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

activation_names = ["linear", "relu", "sigmoid", "tanh", "gelu", "LeakyReLU(0.1)", "ELU", "GELU"]

histories = {}
models = {}
tf.random.set_seed(42)

for name in activation_names:
    model = build_model(name, hidden_dim=64, num_hidden_layers=3)
    print(f"\nTraining with activation: {name}")
    h = model.fit(X_train, y_train, validation_split=0.2,
                  batch_size=64, epochs=200, verbose=0)
    histories[name] = h
    models[name] = model

    y_pred_probs = model.predict(X_test)
    y_pred_classes = (y_pred_probs > 0.5).astype(int)

    # 2. Compute the Evaluation Metrics
    acc = accuracy_score(y_test, y_pred_classes)
    prec = precision_score(y_test, y_pred_classes)
    rec = recall_score(y_test, y_pred_classes)
    f1 = f1_score(y_test, y_pred_classes)

    print(f"Metrics for {activation_name}:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}\n")


### Plots of learning curves

In [ ]:

def plot_learning_curves(history, activation_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Plot Loss
    ax1.plot(history.history['loss'], label='Train Loss')
    ax1.plot(history.history['val_loss'], label='Validation Loss')
    ax1.set_title(f'Loss Curve - {activation_name}')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Binary Crossentropy Loss')
    ax1.legend()
    
    # Plot Accuracy
    ax2.plot(history.history['accuracy'], label='Train Accuracy')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax2.set_title(f'Accuracy Curve - {activation_name}')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

plot_learning_curves(history, "GELU")

In [ ]:
# ==========================================================
# Part D — Plot decision boundaries + training curves
# ==========================================================
def plot_decision_boundary(ax, model, X, y, title=""):
    # grid
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xs = np.linspace(x_min, x_max, 200)
    ys = np.linspace(y_min, y_max, 200)
    XX, YY = np.meshgrid(xs, ys)
    grid = np.c_[XX.ravel(), YY.ravel()].astype(np.float32)
    Z = model.predict(grid, verbose=0).reshape(XX.shape)
    ax.contourf(XX, YY, Z, levels=[0,0.5,1], alpha=0.3)
    ax.scatter(X[:,0], X[:,1], c=y, s=12, edgecolors="none")
    ax.set_title(title)
    ax.set_xlabel("x1"); ax.set_ylabel("x2")

# (1) Decision boundaries
fig, axs = plt.subplots(2, 3, figsize=(14, 8))
for ax, name in zip(axs.ravel(), activation_names):
    plot_decision_boundary(ax, models[name], X_val, y_val, title=name)
plt.suptitle("Effect of activation on learned decision boundary", y=0.95)
plt.tight_layout(); plt.show()

# (2) Loss curves
plt.figure(figsize=(10,5))
for name in activation_names:
    plt.plot(histories[name].history["loss"], label=name)
plt.title("Training loss vs epochs (lower is better)")
plt.xlabel("epoch"); plt.ylabel("binary crossentropy"); plt.legend()
plt.tight_layout(); plt.show()

# (3) Accuracy curves (validation)
plt.figure(figsize=(10,5))
for name in activation_names:
    plt.plot(histories[name].history["val_accuracy"], label=name)
plt.title("Validation accuracy vs epochs")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend()
plt.tight_layout(); plt.show()

# Apply each activation and plot the output distribution
transforms = [
    ("(Linear baseline)", lambda z: z),
    ("ReLU", relu),
    ("Sigmoid", sigmoid),
    ("Tanh", tanh),
    ("LeakyReLU(0.1)", lambda z: leaky_relu(z, 0.1)),
    ("ELU", elu)
]
for i, (name, f) in enumerate(transforms[1:], start=1):
    axs[i].hist(f(Z).ravel(), bins=40)
    axs[i].set_title(name)
    axs[i].set_xlabel("value")
plt.suptitle("How activations reshape the first hidden layer's outputs", y=1.02)
plt.tight_layout(); plt.show()

# ==========================================================
# Key talking points (print for your narration)
# ==========================================================
print("""
Takeaways:
1) Hidden layers learn new feature spaces. With 2 units we could SEE the warping.
2) Nonlinear activations let the model bend decision boundaries:
   • ReLU usually trains fast; simple piecewise linear boundaries.
   • Sigmoid/Tanh can fit well but may learn slower due to saturation (small gradients at extremes).
   • LeakyReLU avoids dead neurons by keeping a small negative slope.
   • ELU smooths negatives and pushes means toward ~0, which can help optimization.
3) Linear baseline struggles on non-linear data: stacking linear layers is still linear → poor boundary.
4) Loss/accuracy curves reveal training dynamics; compare how quickly each activation descends and where it plateaus.
""")
#“What representation does the hidden layer learn for the data?”

In [ ]:
print("""
  - Sigmoid/Tanh can saturate (near 0 or ±1), shrinking gradients.
  - ReLU is simple and fast but 'dies' for x<0 (gradient 0).
  - LeakyReLU keeps small gradient when x<0.
  - ELU smooths negatives and pushes mean activations toward 0, which can speed learning.
""")